In [1]:
!pip install huggingface_hub pyngrok openai-whisper nest_asyncio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 15.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=1a8892819ef48b75c4a799dc73a397575355eaa15d8b62dc9d54e0a922d0628e
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
Successfully built openai-whisper


In [20]:
!ngrok config add-authtoken 2f2i5s0cdMFS65gx7UDpaLYyieJ_73i1yKGHQNjVbzouaYPex

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


**Libraries**

In [ ]:
# AI & Deep Learning
from sentence_transformers import SentenceTransformer
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast, AutoModelForCTC, Wav2Vec2Processor
from transformers import pipeline, AutoModelForSpeechSeq2Seq, WhisperProcessor
from peft import PeftModel
import whisper
import torch

# Network
import asyncio
import websockets
from pyngrok import ngrok

# Utilities
import numpy as np
import time
import json
import joblib
import functools
import struct
import os
from datetime import datetime
import scipy.io.wavfile as wavfile
import pandas as pd

**Configurations and Variables**

In [ ]:
# --- CONFIGURATION ---
VAD_SAMPLE_RATE = 16000
VAD_WINDOW = 512
SILENCE_THRESHOLD = 0.3
SILENCE_CHUNKS = int(SILENCE_THRESHOLD / (VAD_WINDOW / VAD_SAMPLE_RATE))
MIN_SENTENCE_LENGTH = 1.0
MAX_SENTENCE_LENGTH = 5.0
PORT = 5001
transcribe_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PBL6/transcribe.csv")
subtitles_df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/PBL6/subtitles.csv")

# Audio Storage Configuration
AUDIO_STORAGE_PATH = "/content/drive/MyDrive/Colab Notebooks/PBL6/audio_recordings"

# Global storage for video recordings
video_recordings = {}

# --- GLOBAL MODEL VARIABLES ---
device = "cuda" if torch.cuda.is_available() else "cpu"
use_fp16 = device == "cuda"
dtype=torch.float16 if use_fp16 else torch.float32

# --- PATH VARIABLES ---
kmeans_path = "/content/drive/MyDrive/Colab Notebooks/PBL6/kmeans.joblib"
nmt_path = "/content/drive/MyDrive/Colab Notebooks/PBL6/finetune_mbart"
sentence_model_path = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
wav2vec_en_path = "patrickvonplaten/wav2vec2-base-timit-demo-colab"
wav2vec_vn_path = "pdabo1607/Vietnamese_Wav2Vec_Finetune_round2"
BASE_WHISPER_MODEL = "openai/whisper-tiny"
WHISPER_LARGE = "openai/whisper-large-v3"
VIE_ADAPTER_WHISPER_PATH = "/content/drive/MyDrive/Colab Notebooks/PBL6/whisper_tiny_vi"
EN_ADAPTER_WHISPER_PATH = "/content/drive/MyDrive/Colab Notebooks/PBL6/whisper_tiny_en"

**Load models**

In [ ]:
# --- LOAD MODELS ---
def load_models():
    global base_whisper_tiny, whisper_tiny_base_pipe, whisper_largev3_pipe
    global finetuned_whisper_vie, finetuned_whisper_vie_processor
    global finetuned_whisper_en, finetuned_whisper_en_processor
    global wav2vec_en_processor, wav2vec_en, wav2vec_vn_processor, wav2vec_vn
    global finetuned_translator, sentence_model, kmeans, vad_model
    global trans_model, trans_tokenizer
    
    print("[Init] Đang load models... Vui lòng đợi.")
    print(f"[Init] Device: {device}")

    # Load models
    try:
        # === VAD Model (Load FIRST) ===
        vad_model, _ = torch.hub.load(repo_or_dir='snakers4/silero-vad',
                                  model='silero_vad',
                                  force_reload=False)
        print("[Init] VAD Model Loaded")
        
        base_whisper_tiny = AutoModelForSpeechSeq2Seq.from_pretrained(
                        BASE_WHISPER_MODEL, torch_dtype=dtype, device_map=device
                    )
        whisper_tiny_base_pipe = pipeline("automatic-speech-recognition", model=BASE_WHISPER_MODEL)
        print("[Init] Base Whisper Tiny Loaded")

        finetuned_whisper_vie= PeftModel.from_pretrained(base_whisper_tiny, VIE_ADAPTER_WHISPER_PATH)
        finetuned_whisper_vie_processor = WhisperProcessor.from_pretrained(VIE_ADAPTER_WHISPER_PATH)
        finetuned_whisper_vie = finetuned_whisper_vie.merge_and_unload()
        print("[Init] Whisper Vie Loaded")

        finetuned_whisper_en= PeftModel.from_pretrained(base_whisper_tiny, EN_ADAPTER_WHISPER_PATH)
        finetuned_whisper_en_processor = WhisperProcessor.from_pretrained(EN_ADAPTER_WHISPER_PATH)
        finetuned_whisper_en = finetuned_whisper_en.merge_and_unload()
        print("[Init] Whisper En Loaded")

        whisper_largev3_pipe = pipeline("automatic-speech-recognition", model=WHISPER_LARGE)
        print("[Init] Whisper Large Loaded")

        wav2vec_en_processor = Wav2Vec2Processor.from_pretrained(wav2vec_en_path)
        wav2vec_en = AutoModelForCTC.from_pretrained(wav2vec_en_path).to(device)
        print("[Init] Wav2Vec2_en Loaded")

        wav2vec_vn_processor = Wav2Vec2Processor.from_pretrained(wav2vec_vn_path)
        wav2vec_vn = AutoModelForCTC.from_pretrained(wav2vec_vn_path).to(device)
        print("[Init] Wav2Vec2_vn Loaded")

        # === Translation Model (Fixed) ===
        try:
            trans_model = MBartForConditionalGeneration.from_pretrained(nmt_path)
            trans_tokenizer = MBart50TokenizerFast.from_pretrained(nmt_path, use_fast=False)
            finetuned_translator = pipeline(
                "translation",
                model=trans_model,
                tokenizer=trans_tokenizer,
                device=0 if device == "cuda" else -1
            )
            print("[Init] mbart Loaded")
        except Exception as e:
            print(f"[Init Warning] mbart loading failed: {e}")
            finetuned_translator = None

        sentence_model = SentenceTransformer(sentence_model_path)
        print("[Init] Sentence Model Loaded")

        kmeans = joblib.load(kmeans_path)
        print("[Init] Kmeans Loaded")
        
        # Print status
        print("\n" + "="*50)
        print("MODEL STATUS:")
        print(f"VAD: {'✅' if vad_model else '❌'}")
        print(f"Wav2Vec: {'✅' if wav2vec_en and wav2vec_vn else '❌'}")
        print(f"Whisper: {'✅' if finetuned_whisper_en and finetuned_whisper_vie else '❌'}")
        print(f"Translator: {'✅' if finetuned_translator else '❌'}")
        print("="*50 + "\n")
        
    except Exception as e:
        print(f"[Init Error] {e}")
        import traceback
        traceback.print_exc()


**Helper Functions**

In [ ]:
def translate(translator, text, is_en):
    try:
        # Check if translator is available
        if translator is None:
            print("[Warning] Translator is None, returning original text")
            return text
            
        # Defined domain of sentence
        domain = kmeans.predict(sentence_model.encode([text]))[0]
        dtext = f"<D{domain}> " + text

        if is_en:
            src_lang = "en_XX"
            tgt_lang = "vi_VN"
        else:
            src_lang = "vi_VN"
            tgt_lang = "en_XX"

        # Get translate result
        result = translator(
            text,
            src_lang=src_lang,
            tgt_lang=tgt_lang,
            max_length=128
        )

        return result[0]['translation_text']
    except Exception as e:
        print(f"[Translation Error] {e}")
        import traceback
        traceback.print_exc()
        return text

In [25]:
def wav2vec_transcribe(audio_np, origin_lang):
    # Transcription audio
    transcription = (wav2vec_en_processor.decode(
                      torch.argmax(
                        wav2vec_en(
                          wav2vec_en_processor(
                            audio_np,
                            sampling_rate=16000,
                            return_tensors="pt",
                            padding=True
                        ).input_values.to(device)).logits, dim=-1)[0])
                     if origin_lang == 0 else
                     wav2vec_vn_processor.decode(
                        torch.argmax(
                          wav2vec_vn(
                            wav2vec_vn_processor(
                              audio_np,
                              sampling_rate=16000,
                              return_tensors="pt",
                              padding=True
                          ).input_values.to(device)).logits, dim=-1)[0]))
    text = transcription.strip()
    return text if text else None

In [ ]:
def whisper_transcribe(audio_np, origin_lang, model_name):
    result = None

    if "tiny" in model_name:
        result = whisper_tiny_base_pipe(
            audio_np,
            generate_kwargs={"language": "en" if origin_lang == 0 else "vi", "task": "transcribe"}
        )

    elif "largev3" in model_name:
        result = whisper_largev3_pipe(
            audio_np,
            generate_kwargs={"language": "en" if origin_lang == 0 else "vi", "task": "transcribe"}
        )

    elif "finetuned" in model_name:
        if origin_lang == 0:
            inputs = finetuned_whisper_en_processor(audio_np, sampling_rate=16000, return_tensors="pt")
            input_features = inputs.input_features.to(device).to(dtype)

            with torch.no_grad():
                predicted_ids = finetuned_whisper_en.generate(
                    input_features,
                    language="en",      # Sets the language
                    task="transcribe",  # Sets the task
                    max_new_tokens=225
                )

            # Decode
            result = finetuned_whisper_en_processor.batch_decode(predicted_ids, skip_special_tokens=True)
            return result[0] if result else None

        elif origin_lang == 1:
            inputs = finetuned_whisper_vie_processor(audio_np, sampling_rate=16000, return_tensors="pt")
            input_features = inputs.input_features.to(device).to(dtype)

            with torch.no_grad():
                predicted_ids = finetuned_whisper_vie.generate(
                    input_features,
                    language="vi",      # Sets the language
                    task="transcribe",  # Sets the task
                    max_new_tokens=225
                )

            # Decode
            result = finetuned_whisper_vie_processor.batch_decode(predicted_ids, skip_special_tokens=True)
            return result[0] if result else None
        else:
            return None

    if result:
        transcript = result["text"]
        return transcript
    return None

# ============ AUDIO STORAGE FUNCTIONS ============

def save_video_recording(video_id, audio_data, start_timestamp, end_timestamp, metadata):
    """
    Saves the full audio recording for a video to disk
    
    Args:
        video_id: YouTube video ID
        audio_data: numpy array of audio samples (float32)
        start_timestamp: video timestamp when recording started (seconds)
        end_timestamp: video timestamp when recording ended (seconds)
        metadata: dict with additional info (origin_lang, target_lang, etc.)
    """
    try:
        # Create storage directory if it doesn't exist
        os.makedirs(AUDIO_STORAGE_PATH, exist_ok=True)
        
        # Generate filename with timestamp
        timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
        filename = f"{video_id}_{timestamp_str}"
        
        # Convert float32 to int16 for WAV file
        audio_int16 = (audio_data * 32767).astype(np.int16)
        
        # Save audio as WAV file
        wav_path = os.path.join(AUDIO_STORAGE_PATH, f"{filename}.wav")
        wavfile.write(wav_path, VAD_SAMPLE_RATE, audio_int16)
        
        # Save metadata as JSON
        metadata_dict = {
            "video_id": video_id,
            "start_timestamp": start_timestamp,
            "end_timestamp": end_timestamp,
            "duration": end_timestamp - start_timestamp,
            "sample_rate": VAD_SAMPLE_RATE,
            "total_samples": len(audio_data),
            "saved_at": timestamp_str,
            **metadata
        }
        
        json_path = os.path.join(AUDIO_STORAGE_PATH, f"{filename}.json")
        with open(json_path, 'w') as f:
            json.dump(metadata_dict, f, indent=2)
        
        print(f"[Storage] 💾 Saved recording for video {video_id}")
        print(f"[Storage]    Audio: {wav_path}")
        print(f"[Storage]    Metadata: {json_path}")
        print(f"[Storage]    Duration: {end_timestamp - start_timestamp:.2f}s")
        print(f"[Storage]    Size: {len(audio_data) * 4 / 1024 / 1024:.2f} MB")
        
        return wav_path, json_path
        
    except Exception as e:
        print(f"[Storage Error] Failed to save recording: {e}")
        import traceback
        traceback.print_exc()
        return None, None

def init_video_recording(video_id):
    """Initialize a new recording entry for a video ID"""
    if video_id not in video_recordings:
        video_recordings[video_id] = {
            "audio_chunks": [],
            "start_timestamp": None,
            "end_timestamp": None,
            "metadata": {}
        }
        print(f"[Storage] 📹 Initialized recording for video: {video_id}")

def append_audio_to_video(video_id, audio_chunk, current_timestamp, metadata=None):
    """Append an audio chunk to the video's recording"""
    if video_id is None:
        return
    
    init_video_recording(video_id)
    
    recording = video_recordings[video_id]
    recording["audio_chunks"].append(audio_chunk)
    
    # Update timestamps
    if recording["start_timestamp"] is None:
        recording["start_timestamp"] = current_timestamp
    recording["end_timestamp"] = current_timestamp
    
    # Update metadata
    if metadata:
        recording["metadata"].update(metadata)

def finalize_video_recording(video_id):
    """Save and cleanup the recording for a video ID"""
    if video_id is None or video_id not in video_recordings:
        return
    
    recording = video_recordings[video_id]
    
    if not recording["audio_chunks"]:
        print(f"[Storage] ⚠️ No audio chunks for video {video_id}, skipping save")
        del video_recordings[video_id]
        return
    
    # Concatenate all audio chunks
    full_audio = np.concatenate(recording["audio_chunks"])
    
    # Save to disk
    save_video_recording(
        video_id,
        full_audio,
        recording["start_timestamp"],
        recording["end_timestamp"],
        recording["metadata"]
    )
    
    # Cleanup
    del video_recordings[video_id]
    print(f"[Storage] ✅ Finalized and cleaned up recording for video: {video_id}")

In [ ]:
def store_transcribe_results(id, text, full_audio=False):
    if full_audio:
        transcribe_df.loc[id, 'full_audio'] = text
        transcribe_df.to_csv("/content/drive/MyDrive/Colab Notebooks/PBL6/transcribe.csv", index=False)
    else:
        if id in transcribe_df.index:
            transcribe_df.loc[id, 'by_chunk'] = transcribe_df.loc[id, 'by_chunk'] + " " + text
        else:
            transcribe_df.loc[id] = [text, None]

In [ ]:
def save_subtitles(id, origin_text, translate_text, start_time, end_time):
    index = len(subtitles_df)
    subtitles_df.loc[index] = [id, origin_text, translate_text, start_time, end_time]
    subtitles_df.to_csv("/content/drive/MyDrive/Colab Notebooks/PBL6/subtitles.csv", index=False)

# ============ SUBTITLE CACHING FUNCTIONS ============

def get_video_metadata(video_id):
    """
    Get the stored metadata for a video (start and end timestamps of recorded audio)
    Returns None if video doesn't exist in storage
    """
    try:
        # Check all JSON files in storage directory
        if not os.path.exists(AUDIO_STORAGE_PATH):
            return None
        
        for filename in os.listdir(AUDIO_STORAGE_PATH):
            if filename.endswith('.json') and filename.startswith(video_id):
                json_path = os.path.join(AUDIO_STORAGE_PATH, filename)
                with open(json_path, 'r') as f:
                    metadata = json.load(f)
                    if metadata.get('video_id') == video_id:
                        return metadata
        
        return None
    except Exception as e:
        print(f"[Cache] Error reading video metadata: {e}")
        return None

def check_timestamp_coverage(video_id, start_time, end_time):
    """
    Check if the requested timestamp range is already covered by stored audio
    
    Returns:
        - "fully_covered": timestamps are within stored range
        - "not_covered": timestamps are outside stored range (new segment)
        - "not_exists": video doesn't exist in storage
    """
    metadata = get_video_metadata(video_id)
    
    if metadata is None:
        return "not_exists"
    
    stored_start = metadata.get('start_timestamp', 0)
    stored_end = metadata.get('end_timestamp', 0)
    
    # Check if requested range is within stored range (with small tolerance)
    tolerance = 1.0  # 1 second tolerance
    if start_time >= (stored_start - tolerance) and end_time <= (stored_end + tolerance):
        return "fully_covered"
    else:
        return "not_covered"

def get_cached_subtitles(video_id, start_time=None, end_time=None):
    """
    Retrieve cached subtitles for a video from subtitles_df
    
    Args:
        video_id: The video ID
        start_time: Optional start timestamp to filter subtitles
        end_time: Optional end timestamp to filter subtitles
    
    Returns:
        List of subtitle dictionaries or None
    """
    try:
        # Filter subtitles for this video ID
        video_subs = subtitles_df[subtitles_df['id'] == video_id]
        
        if video_subs.empty:
            return None
        
        # If time range specified, filter by timestamps
        if start_time is not None and end_time is not None:
            # Get subtitles that overlap with the requested range
            video_subs = video_subs[
                (video_subs['start_time'] <= end_time) & 
                (video_subs['end_time'] >= start_time)
            ]
        
        if video_subs.empty:
            return None
        
        # Convert to list of dictionaries
        subtitles = []
        for _, row in video_subs.iterrows():
            subtitles.append({
                "origin_text": row['origin_text'],
                "translate_text": row['translate_text'],
                "start": row['start_time'],
                "end": row['end_time']
            })
        
        # Sort by start time
        subtitles.sort(key=lambda x: x['start'])
        
        return subtitles
    
    except Exception as e:
        print(f"[Cache] Error retrieving cached subtitles: {e}")
        import traceback
        traceback.print_exc()
        return None

async def send_cached_subtitles(websocket, video_id, subtitles):
    """
    Send all cached subtitles to the frontend with ALREADY_TRANSCRIBED status
    """
    try:
        print(f"[Cache] 📤 Sending {len(subtitles)} cached subtitles for video {video_id}")
        
        # Send a message indicating we're using cached data
        cache_message = {
            "type": "ALREADY_TRANSCRIBED",
            "video_id": video_id,
            "subtitle_count": len(subtitles),
            "subtitles": subtitles
        }
        
        await websocket.send(json.dumps(cache_message))
        print(f"[Cache] ✅ Sent cached subtitles to frontend")
        
    except Exception as e:
        print(f"[Cache] ❌ Error sending cached subtitles: {e}")
        import traceback
        traceback.print_exc()

def should_use_cache(video_id, start_time, end_time):
    """
    Determine if we should use cached data or run inference
    
    Returns:
        - ("use_cache", subtitles): Use cached subtitles
        - ("run_inference", None): Run normal inference
        - ("new_segment", None): New segment for existing video
    """
    # Check if video exists
    coverage = check_timestamp_coverage(video_id, start_time, end_time)
    
    if coverage == "not_exists":
        print(f"[Cache] ❌ Video {video_id} not in cache - running normal inference")
        return ("run_inference", None)
    
    elif coverage == "fully_covered":
        # Check if we have subtitles for this range
        cached_subs = get_cached_subtitles(video_id, start_time, end_time)
        
        if cached_subs and len(cached_subs) > 0:
            print(f"[Cache] ✅ Found {len(cached_subs)} cached subtitles for video {video_id}")
            print(f"[Cache]    Range: {start_time:.2f}s - {end_time:.2f}s")
            return ("use_cache", cached_subs)
        else:
            print(f"[Cache] ⚠️ Range covered but no subtitles found - running inference")
            return ("run_inference", None)
    
    else:  # not_covered
        print(f"[Cache] 🆕 New segment detected for video {video_id}")
        print(f"[Cache]    Requested: {start_time:.2f}s - {end_time:.2f}s")
        metadata = get_video_metadata(video_id)
        if metadata:
            print(f"[Cache]    Stored: {metadata['start_timestamp']:.2f}s - {metadata['end_timestamp']:.2f}s")
        return ("new_segment", None)

**Main Logics**

In [ ]:
# --- ASR + TRANSLATION LOGIC ---
def run_inference_sync(audio_np, start_offset, duration, startClock, origin_lang, target_lang, video_id, transcription_model, translation_model="mbart"):
    try:
        print(f"[Inference] 🎤 Start - Model: {transcription_model}, Lang: {origin_lang}->{target_lang}, Duration: {duration:.2f}s")
        
        # Transcribe based on model selection
        if transcription_model.startswith("whisper"):
            text = whisper_transcribe(audio_np, origin_lang, transcription_model)
        elif transcription_model.startswith("wav2vec"):
            text = wav2vec_transcribe(audio_np, origin_lang)
        else:
            print(f"[Warning] Unknown model: {transcription_model}, using wav2vec")
            text = wav2vec_transcribe(audio_np, origin_lang)

        if not text:
            print("[Inference] ❌ No text transcribed")
            return None
        
        store_transcribe_results(video_id, text)

        print(f"[Inference] 📝 Transcribed: '{text}'")

        # Check if translator is available
        if finetuned_translator is None:
            print("[Warning] ⚠️ Translator not loaded, returning original text")
            return {
                "type": "transcription",
                "text": text,
                "start": start_offset,
                "end": start_offset + duration,
                "startClock": startClock
            }

        # Translate
        is_en = True if origin_lang == 0 else False
        print(f"[Inference] 🔄 Translating...")
        
        if translation_model == "mbart" or translation_model == "mbartv2":
            final_text = translate(finetuned_translator, text, is_en)
        else:
            final_text = translate(finetuned_translator, text, is_en)

        # Calculate timestamp
        end_offset = start_offset + duration
        print(f"[ASR] ✅ {start_offset:.2f}s -> {end_offset:.2f}s: '{final_text}'")

        save_subtitles(video_id, text, final_text, start_offset, end_offset)

        # Return transcription instantly when source language same as target language
        if target_lang == origin_lang:
            print(f"[Inference] ✅ Same language, no translation needed")
            return {
                "type": "transcription",
                "text": text,
                "start": start_offset,
                "end": end_offset,
                "startClock": startClock
            }

        return {
            "type": "transcription",
            "text": final_text,
            "start": start_offset,
            "end": end_offset,
            "startClock": startClock
        }

    except Exception as e:
        print(f"[Inference Error] ❌ {e}")
        import traceback
        traceback.print_exc()
        return None

In [ ]:
# Network + Audio logics
class StreamSession:
    def __init__(self, websocket, vad_model):
        self.ws = websocket
        self.vad_model = vad_model

        # Buffers & VAD State
        self.sentence_buffer = []
        self.silence_counter = 0
        self.is_speaking = False

        # Sync State
        self.anchor_video_time = 0.0
        self.samples_since_anchor = 0
        self.playback_rate = 1.0
        self.sentence_start_video_time = 0.0

        # Metadata
        self.startClock = 0.0
        self.origin_lang = 0
        self.target_lang = 0
        self.video_id = None
        self.previous_video_id = None  # Track video changes

        # Model Configuration
        self.transcription_model = "wav2vec"
        self.translation_model = "mbart"

    def update_sync(self, data):
        """Handles JSON control messages"""
        if data.get('type') == 'time_sync':
            self.anchor_video_time = float(data['timestamp'])
            self.samples_since_anchor = 0
        elif data.get('type') == 'playback_rate':
            # Update anchor based on how much passed at old speed
            current_offset = (self.samples_since_anchor / VAD_SAMPLE_RATE) * self.playback_rate
            self.anchor_video_time += current_offset
            self.samples_since_anchor = 0
            self.playback_rate = float(data['rate'])
            print(f"[Sync] ⚡ Speed set to {self.playback_rate}x")
        elif data.get('type') == 'config':
            # Update model configuration
            self.transcription_model = data.get('asrModel', 'wav2vec')
            self.translation_model = data.get('translationModel', 'mbart')
            print(f"[Config] 🔧 Transcription: {self.transcription_model}, Translation: {self.translation_model}")
        elif data.get('type') == 'video_changed':
            # Handle video change notification
            new_video_id = data.get('videoId')
            if self.video_id and self.video_id != new_video_id:
                print(f"[Session] 🎬 Video changed from {self.video_id} to {new_video_id}")
                # Finalize previous video recording
                finalize_video_recording(self.video_id)
            self.video_id = new_video_id
            self.previous_video_id = self.video_id

    def parse_audio_message(self, message):
        """Parses binary header and converts audio to Tensor"""
        self.origin_lang = message[0]
        self.target_lang = message[1]
        self.startClock = struct.unpack('<Q', message[2:10])[0]
        
        # Extract video ID length (2 bytes at position 10-11)
        video_id_length = struct.unpack('<H', message[10:12])[0]
        
        # Extract video ID string (UTF-8 encoded)
        video_id_bytes = message[12:12+video_id_length]
        self.video_id = video_id_bytes.decode('utf-8') if video_id_length > 0 else None
        
        print(f"[Audio] 🎧 Origin Lang: {self.origin_lang}, Target Lang: {self.target_lang}, Video ID: {self.video_id}")

        # Audio data starts after the video ID
        audio_data = message[12+video_id_length:]
        audio_int16 = np.frombuffer(audio_data, dtype=np.int16)
        audio_float32 = audio_int16.astype(np.float32) / 32768.0
        return torch.from_numpy(audio_float32)

    def process_vad(self, chunk, current_video_time):
        """Runs VAD on a single chunk and manages the buffer.
           Returns True if a sentence should be triggered."""

        vad_prob = self.vad_model(chunk.unsqueeze(0), VAD_SAMPLE_RATE).item()
        buffer_duration = (len(self.sentence_buffer) * VAD_WINDOW) / VAD_SAMPLE_RATE
        should_trigger = False

        if vad_prob > 0.7:  # Speech
            # Start speaking
            if not self.is_speaking:
                # Get the metadata of start time
                self.sentence_start_video_time = current_video_time

            # Start counter
            self.is_speaking = True
            self.silence_counter = 0
            self.sentence_buffer.append(chunk)

            # Check if buffer has reached max_duration yet
            if buffer_duration >= MAX_SENTENCE_LENGTH:
                should_trigger = True

        else:  # Silence
            # If detect silence while speaking, increase the counter
            if self.is_speaking:
                self.sentence_buffer.append(chunk)
                self.silence_counter += 1

                # If counter reach the limit and buffer has reached the required min sentence length, trigger to cut off sentence
                if self.silence_counter >= SILENCE_CHUNKS and buffer_duration >= MIN_SENTENCE_LENGTH:
                    should_trigger = True

        return should_trigger

    def get_audio_package(self):
        """Prepares the data for inference and resets buffers"""
        full_audio = torch.cat(self.sentence_buffer).numpy()
        speech_duration = (len(full_audio) / VAD_SAMPLE_RATE) * self.playback_rate

        # Snapshot current metadata
        package = {
            "audio": full_audio,
            "start_time": self.sentence_start_video_time,
            "duration": speech_duration,
            "clock": self.startClock,
            "src_lang": self.origin_lang,
            "tgt_lang": self.target_lang,
            "video_id": self.video_id,
            "transcription_model": self.transcription_model,
            "translation_model": self.translation_model
        }

        # Reset
        self.sentence_buffer = []
        self.is_speaking = False
        self.silence_counter = 0

        return package

**Processing Thread**

In [ ]:
# ASR + Translate audio
async def ASR_translate_task(websocket, package):
    """Runs inference in thread pool and sends result"""
    loop = asyncio.get_running_loop()
    result = await loop.run_in_executor(
        None,
        # ASR + Translate cut off audio return by VAD logic
        functools.partial(
            run_inference_sync,
            package['audio'],
            package['start_time'],
            package['duration'],
            package['clock'],
            package['src_lang'],
            package['tgt_lang'],
            package['video_id'],
            package['transcription_model'],
            package['translation_model']
        )
    )
    if result:
        await websocket.send(json.dumps(result))

In [ ]:
# Websocket client's package handler
async def package_handler(websocket):
    print("[WebSocket] Client connected")
    session = StreamSession(websocket, vad_model) # Initialize State
    
    audio_packet_count = 0
    total_bytes_received = 0
    cache_sent = False  # Track if we've sent cached data for this video

    try:
        async for message in websocket:

            # Control Messages
            if isinstance(message, str):
                try:
                    data = json.loads(message)
                    print(f"[WebSocket] 📋 Control message: {data.get('type')}")
                    session.update_sync(data)
                    
                    # Reset cache flag on video change
                    if data.get('type') == 'video_changed':
                        cache_sent = False
                        
                except json.JSONDecodeError:
                    print(f"[WebSocket] ⚠️ Invalid JSON: {message[:50]}")
                    pass
                continue

            # Audio Processing
            audio_packet_count += 1
            total_bytes_received += len(message)
            
            # Log every 10 packets
            if audio_packet_count % 10 == 0:
                print(f"[WebSocket] 🎵 Audio packets received: {audio_packet_count}, Total: {total_bytes_received/1024:.1f} KB")
            
            try:
                audio_tensor = session.parse_audio_message(message)
                number_of_chunks = len(audio_tensor) // VAD_WINDOW
                
                # Check for video change
                if session.video_id != session.previous_video_id:
                    if session.previous_video_id is not None:
                        print(f"[Session] 🎬 Video ID changed: {session.previous_video_id} -> {session.video_id}")
                        finalize_video_recording(session.previous_video_id)
                    session.previous_video_id = session.video_id
                    cache_sent = False  # Reset cache flag for new video
                
                # CHECK CACHE on first packet for this video
                if audio_packet_count == 1 and session.video_id and not cache_sent:
                    print(f"[WebSocket] 🔍 Checking cache for video {session.video_id}...")
                    
                    # Calculate approximate timestamp range (we'll check full video)
                    current_time = session.anchor_video_time
                    
                    # Check if we should use cached data
                    cache_status, cached_subs = should_use_cache(
                        session.video_id, 
                        current_time, 
                        current_time + 3600  # Check up to 1 hour ahead
                    )
                    
                    if cache_status == "use_cache" and cached_subs:
                        # Send cached subtitles immediately
                        await send_cached_subtitles(websocket, session.video_id, cached_subs)
                        cache_sent = True
                        # Continue processing but skip inference (we'll still record audio)
                    elif cache_status == "not_exists":
                        print(f"[Cache] 📝 New video - will run normal inference")
                    elif cache_status == "new_segment":
                        print(f"[Cache] 📝 New segment - will run normal inference and append")
                
                if audio_packet_count == 1:
                    print(f"[WebSocket] ✅ First audio packet processed: {len(audio_tensor)} samples, {number_of_chunks} chunks")

                for i in range(number_of_chunks):
                    # Slicing audio into smaller chunk
                    start = i * VAD_WINDOW
                    end = start + VAD_WINDOW
                    chunk = audio_tensor[start:end]

                    # Calc Time
                    current_time = session.anchor_video_time + \
                        ((session.samples_since_anchor + start) / VAD_SAMPLE_RATE) * session.playback_rate

                    # Run VAD Logic
                    should_trigger = session.process_vad(chunk, current_time)

                    # Fire Inference Task (skip if we already sent cached data)
                    if should_trigger and not cache_sent:
                        print(f"[VAD] 🔊 Speech detected! Triggering inference at {current_time:.2f}s")
                        package = session.get_audio_package()
                        
                        # Check cache for this specific segment
                        segment_start = package['start_time']
                        segment_end = package['start_time'] + package['duration']
                        cache_status, cached_subs = should_use_cache(
                            session.video_id,
                            segment_start,
                            segment_end
                        )
                        
                        if cache_status == "use_cache" and cached_subs:
                            # This specific segment is cached, send it
                            print(f"[Cache] ✅ Using cached subtitle for segment {segment_start:.2f}s - {segment_end:.2f}s")
                            for sub in cached_subs:
                                await websocket.send(json.dumps({
                                    "type": "transcription",
                                    "text": sub["translate_text"],
                                    "start": sub["start"],
                                    "end": sub["end"],
                                    "startClock": package['clock']
                                }))
                        else:
                            # Run inference for this segment
                            asyncio.create_task(ASR_translate_task(websocket, package))

                # Store full audio chunk to video recording (always do this)
                audio_np = audio_tensor.numpy()
                current_time = session.anchor_video_time + \
                    (session.samples_since_anchor / VAD_SAMPLE_RATE) * session.playback_rate
                
                metadata = {
                    "origin_lang": session.origin_lang,
                    "target_lang": session.target_lang,
                    "transcription_model": session.transcription_model,
                    "translation_model": session.translation_model
                }
                append_audio_to_video(session.video_id, audio_np, current_time, metadata)

                # Update counter after processing all chunks in this message
                session.samples_since_anchor += len(audio_tensor)
                
            except Exception as e:
                print(f"[WebSocket] ❌ Error processing audio packet: {e}")
                import traceback
                traceback.print_exc()

    except websockets.exceptions.ConnectionClosed:
        print("\n[WebSocket] Client disconnected")
        # Save any remaining recording
        if session.video_id:
            finalize_video_recording(session.video_id)
    except Exception as e:
        print(f"\n[Error] ❌ {e}")
        import traceback
        traceback.print_exc()
        # Save any remaining recording
        if session.video_id:
            finalize_video_recording(session.video_id)

**Main Thread**

In [31]:
# --- MAIN ENTRY ---
async def main():
    # 1. Load models trước
    load_models()

    # 2. Setup Ngrok
    public_url = ngrok.connect(PORT).public_url
    print(f" * ngrok tunnel \"{public_url}\" -> \"ws://127.0.0.1:{PORT}\"")
    print(f" * CLIENT CONNECT URL: {public_url.replace('https', 'wss').replace('http', 'ws')}")

    # 3. Start Server
    print(f"[Main] Starting WebSocket Server on port {PORT}...")
    async with websockets.serve(package_handler, "localhost", PORT):
        await asyncio.Future()  # Run forever

if __name__ == "__main__":
    # Run
    try:
        asyncio.run(main())
    except RuntimeError as e:
        # Hỗ trợ Colab/Jupyter nếu cần
        if "running event loop" in str(e):
            import nest_asyncio
            nest_asyncio.apply()
            asyncio.run(main())
        else:
            raise e

[Init] Đang load models... Vui lòng đợi.
[Init] Device: cuda


Device set to use cuda:0


[Init] Base Whisper Tiny Loaded
[Init] Whisper Vie Loaded


/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:282: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


[Init] Whisper En Loaded


Device set to use cuda:0


[Init] Whisper Large Loaded
[Init] Wav2Vec2_en Loaded
[Init] Wav2Vec2_vn Loaded
[Init Warning] Không thể load model dịch (kiểm tra lại đường dẫn): 'dict' object has no attribute 'model_type'
 * ngrok tunnel "https://808c735e6f61.ngrok-free.app" -> "ws://127.0.0.1:5001"
 * CLIENT CONNECT URL: wss://808c735e6f61.ngrok-free.app
[Main] Starting WebSocket Server on port 5001...
[WebSocket] Client connected
[Config] 🔧 Transcription: whisper_finetuned, Translation: mbart

[Error] 'NoneType' object is not callable


KeyboardInterrupt: 